# Abdomen CT — nnUNet v2 — **Faz 2a: Preprocessing**
### Kaggle Notebook (Birincil) · Google Colab (İkincil)

**Bu notebook sadece preprocessing'i çalıştırır ve çıktıları paketler.**  
Eğitim için → **Faz2b_Train_Eval_Kaggle.ipynb**

**Kaggle'da çalıştırma:**
1. Sağ panelden Dataset ekle: `ramazan2020/abdomen-acikveri`
2. `Settings → Accelerator → GPU (P100/T4)` seç
3. Hücreleri sırayla çalıştır — son hücre çıktıları `/kaggle/working`'e paketler
4. **Save Version** → output'u `nnunet-preprocessed` dataset olarak yayınlayın

---

| Adım | Süre (T. yakl.) |
|------|-----------------|
| DICOM → NIfTI | 20–40 dk |
| Plan + Preprocess | 15–30 dk |
| Paketleme (working'e kopyalama) | 5–10 dk |

---
## 0. Ortam Tespiti ve GPU Kontrolü

In [1]:
import os, sys, subprocess
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB  = not IS_KAGGLE and os.path.exists('/content')
IS_LOCAL  = not IS_KAGGLE and not IS_COLAB

env_name = 'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'Local'
print(f"Ortam : {env_name}")

import torch
if not torch.cuda.is_available():
    if IS_LOCAL and torch.backends.mps.is_available():
        print("GPU   : Apple Silicon MPS (nnUNet train MPS'i desteklemez; preprocess/NIfTI çalışır)")
    else:
        raise RuntimeError(
            "GPU bulunamadı!\n"
            "Kaggle: Settings → Accelerator → GPU\n"
            "Colab : Runtime → Change runtime type → GPU"
        )
else:
    print(f"GPU   : {torch.cuda.get_device_name(0)}")
    print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    print(f"PyTorch CUDA: {torch.version.cuda}")

Ortam : Local
GPU   : Apple Silicon MPS (nnUNet train MPS'i desteklemez; preprocess/NIfTI çalışır)


---
## 1. Ortam Kurulumu

- **Kaggle**: Sadece kütüphane kurulumu
- **Colab**: Kaggle API kimlik doğrulama + Google Drive bağlantısı

In [2]:
if IS_COLAB:
    # ── Kaggle API ─────────────────────────────────────────────────────────
    # Colab sidebar'dan 🔑 simgesine tıklayın ve KAGGLE_USERNAME, KAGGLE_KEY ekleyin
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'], check=True)
    try:
        from google.colab import userdata
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
        print("Kaggle kimlik bilgileri Colab Secrets'tan yüklendi")
    except Exception:
        from google.colab import files
        import json as _j, shutil as _sh
        print("kaggle.json dosyasını seçin:")
        _up = files.upload()
        _kp = Path.home() / '.kaggle' / 'kaggle.json'
        _kp.parent.mkdir(parents=True, exist_ok=True)
        for fname in _up:
            _sh.move(fname, str(_kp))
        os.chmod(str(_kp), 0o600)
        _kd = _j.loads(_kp.read_text())
        os.environ['KAGGLE_USERNAME'] = _kd['username']
        os.environ['KAGGLE_KEY']      = _kd['key']
        print("kaggle.json yüklendi")

    # ── Google Drive ────────────────────────────────────────────────────────
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print("Google Drive bağlandı")

else:
    print("Kaggle ortamı — API kurulumu ve Drive atlandı")

Kaggle ortamı — API kurulumu ve Drive atlandı


---
## 2. Kütüphane Kurulumu

In [3]:
print("Kütüphaneler kuruluyor...")
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'nnunetv2', 'SimpleITK', 'pydicom', 'scipy', 'tqdm'],
    check=True
)
import importlib; importlib.invalidate_caches()

import nnunetv2, SimpleITK, pydicom, scipy, tqdm

import shutil, sysconfig
def find_nnunet_cmd(name: str) -> str:
    p = shutil.which(name)
    if p: return p
    for d in [sysconfig.get_path('scripts'), str(Path(sys.executable).parent),
              '/usr/local/bin', '/opt/conda/bin']:
        c = Path(d) / name
        if c.exists(): return str(c)
    return name

for cmd in ['nnUNetv2_plan_and_preprocess', 'nnUNetv2_train', 'nnUNetv2_predict']:
    p = find_nnunet_cmd(cmd)
    print(f"  {cmd}: {Path(p).exists()}")

Kütüphaneler kuruluyor...
  nnUNetv2_plan_and_preprocess: True
  nnUNetv2_train: True
  nnUNetv2_predict: True


---
## 3. Konfigürasyon

**Kaggle**: Dataset slug'ını `KAGGLE_DATASET_SLUG` ile belirtin (input dizinindeki klasör adı).  
**Colab**: Dataset Kaggle'dan indirilir, Drive'a preprocessed kaydedilir.

In [4]:
IS_KAGGLE

False

In [5]:
import os, sys, json, shutil, time, subprocess
import numpy as np
import pandas as pd
from pathlib import Path
from typing import List

# ─── Kullanıcı Ayarları ────────────────────────────────────────────────────
KAGGLE_DATASET_SLUG = 'abdomen'
KAGGLE_DATASET_ID   = 'ramazan2020/abdomen'
FOLD         = 0
DATASET_ID   = 100
DATASET_NAME = 'Abdomen'
GITHUB_URL   = 'https://github.com/ramazan2020/abdomen1.git'
# ──────────────────────────────────────────────────────────────────────────


In [6]:

SUPER_CLASSES: List[str] = [
    'acute_cholecystitis', 'kidney_ureter_stone', 'acute_pancreatitis',
    'aortic_aneurysm_dissection', 'acute_appendicitis', 'acute_diverticulitis',
]

if IS_KAGGLE:
    DATA_DIR    = Path('/kaggle/input/datasets/ramazan2020') / KAGGLE_DATASET_SLUG
    WORK_DIR    = Path('/kaggle/working')
    TMP_DIR     = Path('/kaggle/tmp')   # 60 GB — NIfTI + raw (geçici)
    NIFTI_DIR             = TMP_DIR  / 'nifti'
    NNUNET_RAW            = TMP_DIR  / 'nnunet_raw'
    NNUNET_PREPROCESSED_P = WORK_DIR / 'nnunet_preprocessed'  # /kaggle/working → dataset output'a dahil
    NNUNET_RESULTS_P      = WORK_DIR / 'nnunet_results'
    _checkpoint_dataset   = Path('/kaggle/input/nnunet-checkpoint')
    CHECKPOINT_INPUT = _checkpoint_dataset if _checkpoint_dataset.exists() else None

elif IS_COLAB:
    DATA_DIR    = Path('/content/kaggle_data')
    DRIVE_BASE  = Path('/content/drive/MyDrive/Abdomen')
    NIFTI_DIR   = Path('/content/nifti')
    NNUNET_RAW            = DRIVE_BASE / 'nnunet_raw'
    NNUNET_PREPROCESSED_D = DRIVE_BASE / 'nnunet_preprocessed'
    NNUNET_PREPROCESSED_L = Path('/content/nnunet_preprocessed')
    NNUNET_RESULTS_D      = DRIVE_BASE / 'nnunet_results'
    NNUNET_RESULTS_L      = Path('/content/nnunet_results')
    NNUNET_PREPROCESSED_P = NNUNET_PREPROCESSED_L
    NNUNET_RESULTS_P      = NNUNET_RESULTS_L
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)

elif IS_LOCAL:
    try:
        from dotenv import load_dotenv; load_dotenv()
    except ImportError:
        pass
    _proje   = Path(os.environ.get('TR_ABDOMEN_PROJE',  r'D:/makale-pdf/Proje'))
    DATA_DIR = Path(os.environ.get('TR_ABDOMEN_BASE',   str(_proje / 'abdomen')))
    WORK_DIR = Path(os.environ.get('ABDOMEN_OUT_DIR',   str(_proje / 'outputs')))
    DRIVE_BASE = None
    NIFTI_DIR             = WORK_DIR / 'nifti'
    NNUNET_RAW            = WORK_DIR / 'nnunet_raw'
    NNUNET_PREPROCESSED_P = WORK_DIR / 'nnunet_preprocessed'
    NNUNET_RESULTS_P      = WORK_DIR / 'nnunet_results'

EGITIM_DIR  = Path(os.environ.get('ABDOMEN_TRAIN_DIR', str(DATA_DIR / 'Egitim Verisi')))
YARISMA_DIR = Path(os.environ.get('ABDOMEN_TEST_DIR',  str(DATA_DIR / 'Test Verisi')))
BILGI_XLSX    = Path(os.environ.get("ABDOMEN_BILGI_XLSX", str(DATA_DIR / "Bilgi.xlsx")))

SPLIT_DIR   = Path(os.environ.get('ABDOMEN_SPLIT_DIR', str(WORK_DIR / 'splits')))

DATASET_DIR = NNUNET_RAW / f'Dataset{DATASET_ID}_{DATASET_NAME}'

for p in [
    NIFTI_DIR, NNUNET_RAW, NNUNET_PREPROCESSED_P, NNUNET_RESULTS_P,
    DATASET_DIR / 'imagesTr', DATASET_DIR / 'labelsTr',
    SPLIT_DIR
]:
    p.mkdir(parents=True, exist_ok=True)

os.environ['nnUNet_raw']          = str(NNUNET_RAW)
os.environ['nnUNet_preprocessed'] = str(NNUNET_PREPROCESSED_P)
os.environ['nnUNet_results']      = str(NNUNET_RESULTS_P)
os.environ['OMP_NUM_THREADS']     = '1'
os.environ['ABDOMEN_SPLIT_DIR']   = str(SPLIT_DIR)
os.environ['ABDOMEN_TRAIN_DIR']   = str(EGITIM_DIR)
os.environ['ABDOMEN_TEST_DIR']    = str(YARISMA_DIR)
os.environ['ABDOMEN_OUT_DIR']     = str(WORK_DIR / 'abdomen_src_out')
os.environ['ABDOMEN_BILGI_XLSX']     = str(BILGI_XLSX)


print(f'Ortam      : {env_name}')
print(f'DATA_DIR   : {DATA_DIR}  (var={DATA_DIR.exists()})')
print(f'EGITIM_DIR : {EGITIM_DIR}  (var={EGITIM_DIR.exists()})')
print(f'SPLIT_DIR  : {SPLIT_DIR}  (var={SPLIT_DIR.exists()})')
if IS_KAGGLE:
    print(f'TMP_DIR    : {TMP_DIR}  (NIfTI + nnUNet raw/preprocessed)')

if IS_KAGGLE and not EGITIM_DIR.exists():
    raise FileNotFoundError(
        f'Dataset bulunamadı: {DATA_DIR}\n'
        f'Kaggle sağ panelden Datasets → "{KAGGLE_DATASET_SLUG}" ekleyin.'
    )


Ortam      : Local
DATA_DIR   : /Users/ramazanpolat/Desktop/datasets/abdomenDataSet  (var=True)
EGITIM_DIR : /Users/ramazanpolat/Desktop/datasets/abdomenDataSet/Egitim Verisi  (var=True)
SPLIT_DIR  : /Users/ramazanpolat/Desktop/datasets/abdomen/outputs/splits  (var=True)


In [7]:
BILGI_XLSX

PosixPath('/Users/ramazanpolat/Desktop/datasets/abdomenDataSet/Bilgi.xlsx')

---
## 4. Veri İndirme (Yalnızca Colab)

**Kaggle'da bu hücre otomatik atlanır.**

In [8]:
if IS_KAGGLE:
    print("Kaggle: Dataset zaten mount edilmiş, indirme atlandı")
    print(f"  Egitim Verisi: {len(list(EGITIM_DIR.iterdir()))} vaka")
elif IS_COLAB:
    # Colab: Kaggle'dan indir
    _already = EGITIM_DIR.exists() and any(EGITIM_DIR.iterdir())
    if _already:
        print(f"Veri zaten mevcut: {len(list(EGITIM_DIR.iterdir()))} vaka")
    else:
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        print(f"Dataset indiriliyor: {KAGGLE_DATASET_ID}")
        print("Bu işlem 30-90 dakika sürebilir...")
        t0 = time.time()
        r = subprocess.run(
            ['kaggle', 'datasets', 'download',
             '-d', KAGGLE_DATASET_ID,
             '-p', str(DATA_DIR), '--unzip'],
            capture_output=True, text=True
        )
        if r.returncode != 0:
            print('HATA:', r.stderr[-1000:])
            raise RuntimeError('İndirme başarısız')
        print(f"İndirildi ({time.time()-t0:.0f}s)")

    # Colab: preprocessed Drive'da varsa lokal kopyasını al
    _drive_pre = NNUNET_PREPROCESSED_D / f'Dataset{DATASET_ID}_{DATASET_NAME}'
    _local_pre = NNUNET_PREPROCESSED_L / f'Dataset{DATASET_ID}_{DATASET_NAME}'
    if _drive_pre.exists() and not _local_pre.exists():
        print('Preprocessed Drive→Local kopyalanıyor...')
        t0 = time.time()
        shutil.copytree(str(_drive_pre), str(_local_pre))
        print(f'Kopyalandı ({time.time()-t0:.0f}s)')
        os.environ['nnUNet_results'] = str(NNUNET_RESULTS_L)

assert EGITIM_DIR.exists(), f'EGITIM_DIR bulunamadı: {EGITIM_DIR}'
assert SPLIT_DIR.exists(),  f'SPLIT_DIR bulunamadı: {SPLIT_DIR}'
print("Veri doğrulandı")

Veri doğrulandı


In [9]:
# ── GitHub Repo / Local src ───────────────────────────────────────────────
if IS_LOCAL:
    # Local: src/ proje kökünde zaten var, klonlamaya gerek yok
    PROJECT_ROOT = Path(__file__).resolve().parent if '__file__' in dir() else Path('.').resolve()
    REPO_DIR = PROJECT_ROOT
    print(f'Local: src/ kullanılıyor → {REPO_DIR / "src"}')
else:
    REPO_DIR = WORK_DIR / 'abdomen1'
    if not (REPO_DIR / '.git').exists():
        print(f'Klonlanıyor: {GITHUB_URL}')
        subprocess.run(
            ['git', 'clone', '--depth=1', GITHUB_URL, str(REPO_DIR)],
            check=True
        )
    else:
        print('Repo mevcut, güncelleniyor...')
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'],
                       capture_output=True)
    print(f'Repo : {REPO_DIR}')

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from src.splits     import load_fold, raw_case_id
from src.lifting    import BboxLifter
from src.evaluation import f1_at_iou, top5_f1_mean

print('src.splits      ✓')
print('src.lifting     ✓')
print('src.evaluation  ✓')

Local: src/ kullanılıyor → /Users/ramazanpolat/Desktop/datasets/abdomen/src
src.splits      ✓
src.lifting     ✓
src.evaluation  ✓


In [10]:
SPLIT_DIR

PosixPath('/Users/ramazanpolat/Desktop/datasets/abdomen/outputs/splits')

In [11]:
# ── Veri Hazırlık Kontrolü ─────────────────────────────────────────────────
# manifest.csv + splits, Faz1_2_VeriHazirlik notebook'u tarafından üretilir.
# Mevcut değilse burada otomatik oluşturulur.

SPLIT_DIR.mkdir(parents=True, exist_ok=True)
_manifest_csv = SPLIT_DIR / 'manifest.csv'
_splits_csv   = SPLIT_DIR / 'splits.csv'

if not _manifest_csv.exists():
    print('manifest.csv bulunamadı — oluşturuluyor...')
    _bilgi = BILGI_XLSX
    if not _bilgi.exists():
        raise FileNotFoundError(
            f'Bilgi.xlsx bulunamadı: {_bilgi}\n'
            "Faz1_2_VeriHazirlik notebook'unu önce çalıştırın."
        )
    os.environ.setdefault('ABDOMEN_BILGI_XLSX', str(_bilgi))
    from src.preprocessing import build_manifest
    build_manifest(_manifest_csv)
    print(f'manifest.csv oluşturuldu ✓  ({_manifest_csv.stat().st_size/1e3:.0f} KB)')
else:
    print(f'manifest.csv mevcut ✓  ({_manifest_csv.stat().st_size/1e3:.0f} KB)')

if not _splits_csv.exists():
    print('splits.csv bulunamadı — oluşturuluyor...')
    from src.splits import make_splits
    make_splits(out_dir=SPLIT_DIR)
    print('splits.csv oluşturuldu ✓')
else:
    print('splits.csv mevcut ✓')

_fold_csv = SPLIT_DIR / f'fold{FOLD}_train.csv'
if not _fold_csv.exists():
    print(f'fold{FOLD}_train.csv eksik — make_splits yeniden çalıştırılıyor...')
    from src.splits import make_splits
    make_splits(out_dir=SPLIT_DIR)
print(f'fold{FOLD}_train.csv: {"✓ mevcut" if _fold_csv.exists() else "⚠ hâlâ eksik"}')

manifest.csv mevcut ✓  (5083 KB)
splits.csv mevcut ✓
fold0_train.csv: ✓ mevcut


---
## 5. Yardımcı Fonksiyonlar

In [ ]:
manifest = pd.read_csv(SPLIT_DIR / 'manifest.csv')

# BboxLifter: 2D annotasyonları 3D'ye yükseltir (src.lifting)
_lifter = BboxLifter(
    manifest,
    egitim_dir=EGITIM_DIR,
    yarisma_dir=YARISMA_DIR,
)

def lift_2d_to_3d(manifest_df, case, delta_z=2, iou_th=0.3):
    """src.lifting.BboxLifter için geriye uyumlu sarmalayıcı."""
    _l = BboxLifter(manifest_df, egitim_dir=EGITIM_DIR,
                    yarisma_dir=YARISMA_DIR, delta_z=delta_z, iou_th=iou_th)
    return _l.lift(case)

train_cases = load_fold(FOLD, 'train')
val_cases   = load_fold(FOLD, 'val')
all_cases   = sorted(set(train_cases + val_cases))

def case_nii_id(case: str) -> str:
    return case.replace('_', '')

print(f'Manifest  : {len(manifest):,} satır')
print(f'Fold {FOLD} : {len(train_cases)} train + {len(val_cases)} val = {len(all_cases)} toplam')
print('src.lifting.BboxLifter hazır ✓')

Manifest  : 39,268 satır
Fold 0 : 742 train + 186 val = 928 toplam
src.lifting.BboxLifter hazır ✓


In [13]:
# f1_at_iou ve top5_f1_mean src.evaluation'dan import edildi (cell_github)
# İmzalar:
#   f1_at_iou(pred_df, gt_df, iou_th)  → {'per_class', 'macro_f1', 'micro_f1'}
#   top5_f1_mean(pred_df, gt_df)        → {'per_threshold', 'top5', 'top5_mean_f1'}
print('src.evaluation.f1_at_iou     ✓')
print('src.evaluation.top5_f1_mean  ✓')

src.evaluation.f1_at_iou     ✓
src.evaluation.top5_f1_mean  ✓


---
## 6. DICOM → NIfTI Dönüşümü

**Kaggle**: Her session'da yeniden çalışır (working disk session ömürlüdür).  
Paralel dönüşüm — mevcut dosyalar atlanır.

In [14]:
import SimpleITK as sitk
import pydicom
from concurrent.futures import ThreadPoolExecutor
from tqdm.notebook import tqdm

def dicom_to_nifti(case: str) -> str:
    raw = raw_case_id(case)
    out = NIFTI_DIR / f'case_{case_nii_id(case)}_0000.nii.gz'
    if out.exists(): return 'skip'
    case_dir = EGITIM_DIR / str(raw) if case.startswith('T_') else YARISMA_DIR / str(raw)
    if not case_dir.exists(): return f'missing:{case}'
    reader = sitk.ImageSeriesReader()
    names = reader.GetGDCMSeriesFileNames(str(case_dir))
    if not names: return f'no_dcm:{case}'

    # Karışık boyutlu serileri filtrele: baskın (Rows×Cols) grubunu al
    # 608×512 gibi anormal boyutlular genellikle bozuk/truncated DICOM'dur;
    # header boyutu söyler ama pixel verisi farklı boyuttadır → atılmaları güvenlidir
    size_map = {}
    for n in names:
        try:
            h = pydicom.dcmread(n, stop_before_pixels=True)
            size_map.setdefault((int(h.Rows), int(h.Columns)), []).append(n)
        except Exception:
            pass
    if len(size_map) > 1:
        dominant = max(size_map.values(), key=len)
        dropped = sum(len(v) for k, v in size_map.items() if v is not dominant)
        names = dominant
        # Atılan dilim sayısını döndür (eğitim için tanı amaçlı)
        return_suffix = f'|dropped:{dropped}'
    else:
        return_suffix = ''

    reader.SetFileNames(names)
    try:
        sitk.WriteImage(reader.Execute(), str(out))
        return 'ok' + return_suffix
    except Exception as e:
        return f'err:{e}'

_n_workers = min(os.cpu_count() or 4, 8)

print(f'DICOM→NIfTI: {len(all_cases)} vaka')
print(f'  T_ → {EGITIM_DIR}')
print(f'  C_ → {YARISMA_DIR}')
print(f'Workers: {_n_workers}  (CPU: {os.cpu_count()})')
_nii_existing = len(list(NIFTI_DIR.glob('*.nii.gz')))
print(f'Mevcut NIfTI: {_nii_existing}')

sitk.ProcessObject.GlobalWarningDisplayOff()
with ThreadPoolExecutor(max_workers=_n_workers) as ex:
    results = list(tqdm(ex.map(dicom_to_nifti, all_cases),
                        total=len(all_cases), desc='DICOM→NIfTI'))
sitk.ProcessObject.GlobalWarningDisplayOn()

ok      = sum(1 for r in results if r.startswith('ok') and 'dropped' not in r)
ok_filt = [r for r in results if r.startswith('ok|dropped:')]
skip    = sum(1 for r in results if r == 'skip')
errs    = [r for r in results if r.startswith('err:')]

print(f'{ok} yeni  |  {len(ok_filt)} boyut filtreli  |  {skip} atlandı  |  {len(errs)} hata')
if ok_filt:
    for r in ok_filt:
        dropped_n = int(r.split('dropped:')[1])
        # hangi vaka olduğunu bulmak için indekse bak
        idx = results.index(r)
        print(f'  → {all_cases[idx]}: {dropped_n} bozuk dilim atıldı (header/pixel boyut uyumsuzluğu)')
if errs:
    print('Hatalar:', errs[:5])
print(f'Toplam NIfTI: {len(list(NIFTI_DIR.glob("*.nii.gz")))}')


DICOM→NIfTI: 928 vaka
  T_ → /Users/ramazanpolat/Desktop/datasets/abdomenDataSet/Egitim Verisi
  C_ → /Users/ramazanpolat/Desktop/datasets/abdomenDataSet/Test Verisi
Workers: 8  (CPU: 10)
Mevcut NIfTI: 742


DICOM→NIfTI:   0%|          | 0/928 [00:00<?, ?it/s]

290 yeni  |  0 boyut filtreli  |  638 atlandı  |  0 hata
Toplam NIfTI: 1032


---
## 7. nnUNet Dataset Formatı Hazırlama

- `imagesTr/` → CT hacimler (symlink)
- `labelsTr/` → Semantic maske (0=arka plan, 1-6=patoloji sınıfları)
- `dataset.json` → nnUNet meta

In [15]:
from concurrent.futures import ThreadPoolExecutor

manifest = pd.read_csv(SPLIT_DIR / 'manifest.csv')
print(f'Manifest: {len(manifest)} satır  |  Sütunlar: {list(manifest.columns)}')


def prep_case_nnunet(case: str) -> str:
    raw = raw_case_id(case)
    nii_id  = case_nii_id(case)
    src_nii = NIFTI_DIR / f'case_{nii_id}_0000.nii.gz'
    if not src_nii.exists() or src_nii.stat().st_size == 0: return f'corrupt:{case}'

    dst_img = DATASET_DIR / 'imagesTr' / f'case_{nii_id}_0000.nii.gz'
    if not dst_img.exists():
        try: os.symlink(src_nii, dst_img)
        except (OSError, NotImplementedError): shutil.copy2(str(src_nii), str(dst_img))

    dst_lbl = DATASET_DIR / 'labelsTr' / f'case_{nii_id}.nii.gz'
    if dst_lbl.exists(): return 'skip'

    boxes = lift_2d_to_3d(manifest, case)
    img_itk = sitk.ReadImage(str(src_nii))
    shape = sitk.GetArrayFromImage(img_itk).shape  # (Z, Y, X)
    mask = np.zeros(shape, dtype=np.uint8)
    for b in boxes:
        label = int(b['class']) + 1
        mask[int(b['z1']):min(int(b['z2'])+1,shape[0]),
             int(b['y1']):min(int(b['y2'])+1,shape[1]),
             int(b['x1']):min(int(b['x2'])+1,shape[2])] = label
    m = sitk.GetImageFromArray(mask)
    m.CopyInformation(img_itk)
    sitk.WriteImage(m, str(dst_lbl))
    return 'ok'

# Kaggle RAM ~13 GB → 4 worker güvenli; lokal/Colab'da cpu_count kadar artırılabilir
_n_workers = 4 if IS_KAGGLE else min(os.cpu_count() or 4, 8)

print(f'Dataset hazırlanıyor: {len(all_cases)} vaka  |  Workers: {_n_workers}')
with ThreadPoolExecutor(max_workers=_n_workers) as ex:
    results = list(tqdm(
        ex.map(prep_case_nnunet, all_cases),
        total=len(all_cases), desc='Dataset prep'
    ))

ok   = results.count('ok')
skip = results.count('skip')
miss = [r for r in results if 'missing' in r or 'corrupt' in r]
print(f'{ok} yeni  |  {skip} atlandı  |  {len(miss)} eksik NIfTI')
if miss:
    print('  Eksik NIfTI (dönüşüm başarısız olmuş vakalar):', miss[:5])

n_img = len(list((DATASET_DIR/'imagesTr').glob('*.nii.gz')))
n_lbl = len(list((DATASET_DIR/'labelsTr').glob('*.nii.gz')))
print(f'imagesTr: {n_img}  |  labelsTr: {n_lbl}')
if n_img == 0:
    raise RuntimeError('imagesTr boş!')

# numTraining = gerçek dosya sayısı; len(all_cases) ile fark NIfTI dönüşüm hatası demektir
dataset_json = {
    'channel_names': {'0': 'CT'},
    'labels': {'background': 0, **{n: i+1 for i, n in enumerate(SUPER_CLASSES)}},
    'numTraining': n_img,
    'file_ending': '.nii.gz',
}
(DATASET_DIR / 'dataset.json').write_text(json.dumps(dataset_json, indent=2))
if n_img < len(all_cases):
    print(f'⚠ numTraining={n_img} (beklenen {len(all_cases)}; {len(all_cases)-n_img} NIfTI dönüşümü başarısız)')
else:
    print(f'numTraining={n_img} ✓')


Manifest: 39268 satır  |  Sütunlar: ['case', 'image_id', 'source', 'dicom_path', 'super_labels', 'bboxes', 'anatomical_boundary', 'n_bbox_anns', 'n_boundary_anns']
Dataset hazırlanıyor: 928 vaka  |  Workers: 8


Dataset prep:   0%|          | 0/928 [00:00<?, ?it/s]

307 yeni  |  621 atlandı  |  0 eksik NIfTI
imagesTr: 1032  |  labelsTr: 1032
numTraining=1032 ✓


---
## 8. nnUNet Plan + Preprocess

Fingerprint çıkarımı, plan oluşturma ve veriyi nnUNet formatına dönüştürme.  
**Önceki session'da tamamlandıysa bu hücre atlanır.**

In [16]:
import shutil as _sh, json

_plans_file = (NNUNET_PREPROCESSED_P
               / f'Dataset{DATASET_ID}_{DATASET_NAME}'
               / 'nnUNetPlans.json')

_np = 2 if IS_KAGGLE else max(1, (os.cpu_count() or 4) // 2)

# ── Step 1+2: Fingerprint + Plan (preprocessing'den önce ayrı çalışır) ───────
if not _plans_file.exists():
    for step_name, step_args in [
        ('Fingerprint + Integrity',
         [find_nnunet_cmd('nnUNetv2_extract_fingerprint'),
          '-d', str(DATASET_ID), '-npfp', str(_np), '--verify_dataset_integrity']),
        ('Plan',
         [find_nnunet_cmd('nnUNetv2_plan_experiment'),
          '-d', str(DATASET_ID)]),
    ]:
        print(f'\n▶ {step_name}...')
        _proc = subprocess.Popen(
            step_args, env={**os.environ},
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            universal_newlines=True
        )
        for line in _proc.stdout:
            print(line, end='', flush=True)
        _proc.wait()
        if _proc.returncode != 0:
            raise RuntimeError(f'{step_name} başarısız! Kod: {_proc.returncode}')
    print(f'\nPlan oluşturuldu: {_plans_file}')
else:
    print(f'Plans dosyası mevcut, fingerprint+plan atlandı: {_plans_file}')

# ── Disk kontrolü ve Plans.json oto-ayarlama ──────────────────────────────────
_free_gb  = _sh.disk_usage(str(NNUNET_PREPROCESSED_P)).free  / 1e9
_total_gb = _sh.disk_usage(str(NNUNET_PREPROCESSED_P)).total / 1e9
_nifti_gb = sum(f.stat().st_size for f in NIFTI_DIR.glob('*.nii.gz')) / 1e9

print(f'\nDisk : {_free_gb:.1f} GB serbest / {_total_gb:.1f} GB toplam')
print(f'NIfTI: {_nifti_gb:.1f} GB')

# Tahmini preprocessed boyut: case başına ortalama 100–200 MB float32 uncompressed
_est_pre_gb = len(all_cases) * 0.15   # ~150 MB/case (orta tahmin)
print(f'Tahmini preprocessed: {_est_pre_gb:.0f} GB ({len(all_cases)} vaka × ~150 MB)')

_plans_data = json.loads(_plans_file.read_text())
_cfg = _plans_data.get('configurations', {}).get('3d_fullres', {})
_cur_spacing = _cfg.get('spacing', [])
print(f'Mevcut spacing: {[round(s,3) for s in _cur_spacing]}')

# Disk taşmaması için gereken alan: NIfTI + preprocessed + buffer
_needed_gb = _nifti_gb + _est_pre_gb + 5   # 5 GB buffer
if _free_gb < _needed_gb:
    # XY spacing'i yeterli alana göre ölçekle; Z spacing dokunma
    # Hedef: preprocessed ≤ (free - nifti - 5) GB
    _target_pre_gb = max(_free_gb - _nifti_gb - 5, 10)
    _scale = (_est_pre_gb / _target_pre_gb) ** 0.5   # XY iki boyut, kareköke böl
    _scale = min(max(_scale, 1.2), 2.5)              # 1.2x – 2.5x arası sınırla

    _new_spacing = [_cur_spacing[0]] + [round(s * _scale, 4) for s in _cur_spacing[1:]]
    _new_size    = [round(int(v) / _scale) for v in _cfg.get('patch_size', [80, 160, 160])]
    _new_size[0] = _cfg.get('patch_size', [80, 160, 160])[0]   # Z değişmez

    print(f'\n⚠ Disk yetersiz ({_free_gb:.1f} GB < {_needed_gb:.0f} GB gerekli)')
    print(f'  Spacing  {[round(s,3) for s in _cur_spacing]} → {_new_spacing}  (scale ×{_scale:.2f})')
    print(f'  PatchSiz {_cfg.get("patch_size")} → {_new_size}')

    _cfg['spacing']    = _new_spacing
    _cfg['patch_size'] = _new_size
    _plans_data['configurations']['3d_fullres'] = _cfg
    _plans_file.write_text(json.dumps(_plans_data, indent=2))
    print('  nnUNetPlans.json güncellendi ✓')
else:
    print(f'Disk yeterli ({_free_gb:.1f} GB > {_needed_gb:.0f} GB) — plans.json değiştirilmedi')


# ── Patch size geçerlilik kontrolü (decoder skip bağlantısı hatası önleme) ──
# Eski nnUNet planları patch_size=[80,72,72] üretebilir.
# XY'de 5 yarılanma: 72→36→18→9 (tek!), conv 9→5, decoder 5×2=6 ≠ skip=5 → RuntimeError.
# Her boyutu 2^n_yarılanma katına yuvarlayarak çakışmayı önle.
def _fix_patch_for_strides(patch, strides):
    fixed, changed = list(patch), False
    for dim in range(len(fixed)):
        n_halv = sum(1 for s in strides[1:] if s[dim] >= 2)
        req    = 2 ** n_halv
        if fixed[dim] % req != 0:
            new_val = max((fixed[dim] // req) * req, req)
            print(f'  boyut {dim}: patch {fixed[dim]} → {new_val}  (2^{n_halv}={req} katı gerekli)')
            fixed[dim] = new_val
            changed = True
    return fixed, changed

_strides_in_plan = _cfg.get('architecture', {}).get('arch_kwargs', {}).get('strides', [])
_patch_in_plan   = list(_cfg.get('patch_size', []))

if _strides_in_plan and _patch_in_plan:
    _fixed_patch, _patch_changed = _fix_patch_for_strides(_patch_in_plan, _strides_in_plan)
    if _patch_changed:
        print(f'\n⚠  patch_size uyumsuz → düzeltiliyor: {_patch_in_plan} → {_fixed_patch}')
        _cfg['patch_size'] = _fixed_patch
        _plans_data['configurations']['3d_fullres'] = _cfg
        _plans_file.write_text(json.dumps(_plans_data, indent=2))
        print('  nnUNetPlans.json güncellendi ✓')
        # Hatalı patch ile üretilmiş eski preprocess verisini temizle
        _stale = NNUNET_PREPROCESSED_P / f'Dataset{DATASET_ID}_{DATASET_NAME}' / 'nnUNetPlans_3d_fullres'
        if _stale.exists():
            _sh.rmtree(str(_stale))
            print('  Eski preprocessed silindi → yeni patch ile yeniden üretilecek')
    else:
        print(f'patch_size geçerli: {_patch_in_plan}')
else:
    print("patch_size kontrolü atlandı (strides plans.json'da bulunamadı — eski format)")

# ── Step 3: Preprocess (sadece preprocess) ────────────────────────────────────
_pre_done = (NNUNET_PREPROCESSED_P
             / f'Dataset{DATASET_ID}_{DATASET_NAME}'
             / 'nnUNetPlans_3d_fullres').exists()

if _pre_done:
    print('\nPreprocess klasörü mevcut, atlandı.')
else:
    print(f'\n▶ Preprocess başlatılıyor...  Workers: {_np}')
    print('  ~15–30 dakika sürebilir...')

    _proc = subprocess.Popen(
        [find_nnunet_cmd('nnUNetv2_preprocess'),
         '-d', str(DATASET_ID), '-plans_name', 'nnUNetPlans',
         '-c', '3d_fullres', '-np', str(_np)],
        env={**os.environ},
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        universal_newlines=True
    )
    for line in _proc.stdout:
        print(line, end='', flush=True)
    _proc.wait()
    if _proc.returncode != 0:
        raise RuntimeError(f'Preprocess başarısız! Kod: {_proc.returncode}')
    print('\nPreprocess tamamlandı!')

# Colab: preprocessed'i Drive'a yedekle
if IS_COLAB:
    _src = NNUNET_PREPROCESSED_L / f'Dataset{DATASET_ID}_{DATASET_NAME}'
    _dst = NNUNET_PREPROCESSED_D / f'Dataset{DATASET_ID}_{DATASET_NAME}'
    if _src.exists() and not _dst.exists():
        print("Preprocessed Drive'a yedekleniyor...")
        _sh.copytree(str(_src), str(_dst))
        print('Yedeklendi')

# Kaggle: raw artık gerekmez → /kaggle/tmp temizle
if IS_KAGGLE and NNUNET_RAW.exists():
    print(f'\nTemizleniyor: {NNUNET_RAW}')
    _sh.rmtree(str(NNUNET_RAW))
    print('nnunet_raw silindi ✓')


Plans dosyası mevcut, fingerprint+plan atlandı: /Users/ramazanpolat/Desktop/datasets/abdomen/outputs/nnunet_preprocessed/Dataset100_Abdomen/nnUNetPlans.json

Disk : 177.6 GB serbest / 994.6 GB toplam
NIfTI: 84.1 GB
Tahmini preprocessed: 139 GB (928 vaka × ~150 MB)
Mevcut spacing: [2.0, 0.84, 0.84]

⚠ Disk yetersiz (177.6 GB < 228 GB gerekli)
  Spacing  [2.0, 0.84, 0.84] → [2.0, 1.0534, 1.0534]  (scale ×1.25)
  PatchSiz [80, 160, 160] → [80, 128, 128]
  nnUNetPlans.json güncellendi ✓
patch_size kontrolü atlandı (strides plans.json'da bulunamadı — eski format)

Preprocess klasörü mevcut, atlandı.


In [17]:
# Özel fold split yaz (nnUNet auto-split'ini geçersiz kıl)
_splits_dir = NNUNET_PREPROCESSED_P / f'Dataset{DATASET_ID}_{DATASET_NAME}'
_splits_dir.mkdir(parents=True, exist_ok=True)

_splits = [{
    'train': [f'case_{case_nii_id(c)}' for c in train_cases],
    'val'  : [f'case_{case_nii_id(c)}' for c in val_cases],
}]
(_splits_dir / 'splits_final.json').write_text(json.dumps(_splits, indent=2))
print(f'splits_final.json: {len(train_cases)} train, {len(val_cases)} val')

splits_final.json: 742 train, 186 val


---
## 9. Val NIfTI Paketleme

Val vakalarının NIfTI dosyalarını `nifti_val/` klasörüne kopyala — Faz2b inference için.  
**Kaggle**: `/kaggle/working/nifti_val/` → dataset output'a dahil olur  
**Colab**: Drive'a yedeklenir, Faz2b oradan okur

In [18]:
# Val NIfTI'larını nifti_val/ klasörüne kopyala
# Kaggle : /kaggle/working/nifti_val/ → dataset output'a dahil
# Colab  : Drive/nifti_val/ → Faz2b oradan okur
# Local  : nnunet_preprocessed'in yanına
if IS_KAGGLE:
    NIFTI_VAL_OUT = WORK_DIR / 'nifti_val'
elif IS_COLAB:
    NIFTI_VAL_OUT = NNUNET_PREPROCESSED_D.parent / 'nifti_val'   # Drive'da
else:
    NIFTI_VAL_OUT = NNUNET_PREPROCESSED_P.parent / 'nifti_val'

NIFTI_VAL_OUT.mkdir(parents=True, exist_ok=True)

_copied, _skipped, _missing = 0, 0, []
for c in val_cases:
    _nii_id = case_nii_id(c)
    _src = NIFTI_DIR / f'case_{_nii_id}_0000.nii.gz'
    _dst = NIFTI_VAL_OUT / f'case_{_nii_id}_0000.nii.gz'
    if _dst.exists():
        _skipped += 1
        continue
    if _src.exists():
        shutil.copy2(str(_src), str(_dst))
        _copied += 1
    else:
        _missing.append(c)

_ready = len(list(NIFTI_VAL_OUT.glob('*.nii.gz')))
print(f'Val NIfTI: {_copied} kopyalandı  |  {_skipped} mevcut  |  {_ready}/{len(val_cases)} hazır')
if _missing:
    print(f'⚠ {len(_missing)} eksik (DICOM dönüşümü başarısız): '
          f'{_missing[:5]}')
print(f'Çıktı: {NIFTI_VAL_OUT}')


Val NIfTI: 162 kopyalandı  |  24 mevcut  |  307/186 hazır
Çıktı: /Users/ramazanpolat/Desktop/datasets/abdomen/outputs/nifti_val
